# L22 · Human-in-the-loop: aprobar un desembolso antes de ejecutarlo

*Advanced Topics in Analytics II - Generative AI* · Instructor: Sergio A. Mora Pardo

Hasta ahora nuestros agentes solo **leían**: consultaban la cartera, las políticas o la TRM. Un desembolso es distinto: **mueve dinero** y no se puede deshacer con un clic. En **ACME**, la política dice que el monto máximo desembolsado por cliente en un día es 40.000.000 COP, y un analista revisa los desembolsos grandes antes de hacerlos.

*Human-in-the-loop* (HITL) significa que el flujo **se pausa** en un punto, una persona revisa y decide, y el flujo **continúa** desde donde quedó. LangGraph lo hace con `interrupt`, un **checkpointer** que guarda el estado mientras se espera y `Command(resume=...)` para seguir.

## Qué construimos

```mermaid
flowchart LR
    S([START]) --> V["validate<br/>reglas en código"]
    V -->|monto ≤ 20 M| D["disburse<br/>simulado"]
    V -->|monto > 20 M| A["approval<br/>interrupt: espera a una persona"]
    V -->|monto > 40 M| R["reject<br/>sin persona"]
    A -->|aprobar o editar| D
    A -->|rechazar| R
    D & R --> E([END])
```

## Objetivos

1. Pausar un grafo con `interrupt` y leer qué está esperando.
2. Guardar el estado de la pausa con un checkpointer y un `thread_id`.
3. Reanudar con `Command(resume=...)`: aprobar, rechazar o editar.
4. Poner la aprobación en las herramientas de un agente con `HumanInTheLoopMiddleware`.
5. Explicar por qué la aprobación debe estar en el código y no solo en el prompt.

El recorrido realiza **unas 4 llamadas a Claude Haiku**, todas en la sección 5. Las secciones 1 a 4 no llaman al modelo.

## 1. El flujo sin persona

Sin modelo. El estado es una solicitud de desembolso. `validate` aplica las reglas en código: hasta 20.000.000 COP se desembolsa directo; más de 40.000.000 se rechaza, porque supera el tope diario; en medio, hace falta una persona. `disburse` **simula** el desembolso: en un sistema real llamaría a la plataforma de pagos.

In [ ]:
from typing import Literal, TypedDict

from IPython.display import Markdown, display
from langgraph.graph import END, START, StateGraph

AUTO_LIMIT = 20_000_000
DAILY_LIMIT = 40_000_000


class Disbursement(TypedDict, total=False):
    customer: str
    amount_cop: int
    status: str
    reviewer_note: str


def validate(state: Disbursement) -> dict:
    return {"status": "validado"}


def route(state: Disbursement) -> Literal["disburse", "approval", "reject"]:
    if state["amount_cop"] > DAILY_LIMIT:
        return "reject"
    return "approval" if state["amount_cop"] > AUTO_LIMIT else "disburse"


def disburse(state: Disbursement) -> dict:
    # Simulado: aquí se llamaría a la plataforma de pagos.
    return {"status": f"desembolsados {state['amount_cop']:,} COP a la cuenta ACME de {state['customer']}"}


def reject(state: Disbursement) -> dict:
    return {"status": "rechazado: " + state.get("reviewer_note", "supera el monto máximo diario de 40.000.000 COP")}


def show(graph):
    """Dibuja el grafo como Mermaid: JupyterLab lo renderiza sin conexión a internet."""
    display(Markdown(f"```mermaid\n{graph.get_graph().draw_mermaid()}\n```"))

## 2. Pausar con `interrupt`

`interrupt(valor)` detiene el grafo **dentro** del nodo y entrega `valor` a quien lo ejecuta: aquí, lo que la persona necesita para decidir. Cuando se reanuda, `interrupt` devuelve la decisión de la persona y el nodo sigue desde ese punto.

Para pausar hace falta un **checkpointer**, como en L9: guarda el estado mientras se espera, que puede ser un minuto o un día. Cada solicitud vive en su propio `thread_id`.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt


def approval(state: Disbursement) -> Command[Literal["disburse", "reject"]]:
    decision = interrupt({
        "pregunta": "¿Apruebas este desembolso?",
        "cliente": state["customer"],
        "monto_cop": state["amount_cop"],
        "opciones": ["approve", "reject", "edit"],
    })
    if decision["type"] == "approve":
        return Command(goto="disburse")
    if decision["type"] == "edit":
        return Command(goto="disburse", update={"amount_cop": decision["amount_cop"]})
    return Command(goto="reject", update={"reviewer_note": decision.get("note", "rechazado por el analista")})


builder = StateGraph(Disbursement)
for node in (validate, approval, disburse, reject):
    builder.add_node(node.__name__, node)
builder.add_edge(START, "validate")
builder.add_conditional_edges("validate", route)
builder.add_edge("disburse", END)
builder.add_edge("reject", END)
graph = builder.compile(checkpointer=InMemorySaver())
show(graph)

`approval` devuelve un `Command(goto=...)`: decide el siguiente nodo y, si hace falta, actualiza el estado en el mismo paso. Por eso no tiene aristas de salida en el código; la anotación `Command[Literal[...]]` le dice a LangGraph a dónde puede ir para dibujarlo.

Un desembolso pequeño no pasa por la persona:

In [ ]:
small = graph.invoke({"customer": "C-2210", "amount_cop": 1_200_000}, {"configurable": {"thread_id": "d-001"}})
small["status"]

## 3. La pausa y lo que queda guardado

Un desembolso de 25.000.000 COP llega a `approval` y se detiene. `invoke` devuelve el estado hasta ese punto con la clave `__interrupt__`, que trae el valor que pasó `interrupt`. `get_state` muestra, además, **qué nodo sigue**: el grafo está esperando.

In [ ]:
config = {"configurable": {"thread_id": "d-002"}}
paused = graph.invoke({"customer": "C-4415", "amount_cop": 25_000_000}, config)

print("Esperando a la persona:", paused["__interrupt__"][0].value)
state = graph.get_state(config)
print("Siguiente nodo:", state.next)
print("Estado guardado:", state.values)

El proceso podría terminar aquí y reanudarse horas después, siempre que el checkpointer sea persistente. Con `InMemorySaver` se pierde al reiniciar el kernel; en la semana 12, la API usa PostgreSQL con el mismo `thread_id`.

## 4. Reanudar: aprobar, rechazar o editar

`Command(resume=...)` entrega la decisión a `interrupt` y el grafo sigue **desde la pausa**, en el mismo hilo. Probamos las tres decisiones, cada una en su propio hilo.

In [ ]:
def request(thread_id, amount):
    config = {"configurable": {"thread_id": thread_id}}
    graph.invoke({"customer": "C-4415", "amount_cop": amount}, config)
    return config


approved = graph.invoke(Command(resume={"type": "approve"}), request("d-003", 25_000_000))
rejected = graph.invoke(Command(resume={"type": "reject", "note": "falta la declaración de renta"}), request("d-004", 30_000_000))
edited = graph.invoke(Command(resume={"type": "edit", "amount_cop": 20_000_000}), request("d-005", 28_000_000))

for name, result in [("aprobado", approved), ("rechazado", rejected), ("editado", edited)]:
    print(f"{name:9} → {result['status']}")

Cuando se reanuda, el nodo `approval` **vuelve a ejecutarse desde el principio** hasta llegar al `interrupt`, que esta vez devuelve la decisión. Por eso un nodo con `interrupt` no debe hacer nada irreversible **antes** de la pausa: se repetiría. Aquí, `approval` solo pregunta y decide; el desembolso ocurre en otro nodo.

## 5. La aprobación en las herramientas de un agente

En un agente, el modelo decide qué herramienta llamar. Si una de ellas mueve dinero, no basta con escribir en el *system prompt* «pide confirmación»: el modelo puede no hacerlo. `HumanInTheLoopMiddleware` pone la pausa **en el código**: antes de ejecutar las herramientas que indiquemos, el agente se detiene y espera una decisión. Las demás corren sin preguntar.

```mermaid
sequenceDiagram
    participant P as Analista
    participant A as Agente
    participant T as request_disbursement
    P->>A: Desembolsa 25 M al cliente C-4415
    A->>A: el modelo pide request_disbursement
    A-->>P: interrupt: ¿aprobar, editar o rechazar?
    P->>A: Command(resume={"decisions": [...]})
    A->>T: se ejecuta (o no) según la decisión
    A-->>P: respuesta final
```

In [ ]:
from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

# find_dotenv sube por las carpetas hasta encontrar el .env de la raíz
load_dotenv(find_dotenv())

model = ChatAnthropic(
    model="claude-haiku-5-5",
    max_tokens=400,
    thinking={"type": "disabled"},
    timeout=60,
    max_retries=2,
)


@tool
def request_disbursement(customer: str, amount_cop: int) -> str:
    """Desembolsa un crédito aprobado a la cuenta ACME del cliente. Mueve dinero: requiere aprobación."""
    return f"Desembolsados {amount_cop:,} COP a la cuenta ACME de {customer}."


agent = create_agent(
    model,
    tools=[request_disbursement],
    system_prompt="Eres el asistente de operaciones de ACME. Responde en español y en una o dos frases.",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"request_disbursement": True})],
    checkpointer=InMemorySaver(),
)

El analista pide un desembolso. El modelo propone la herramienta y el agente **se detiene antes de ejecutarla** (**1 llamada**). El `interrupt` trae la acción propuesta, con sus argumentos, y las decisiones permitidas.

In [ ]:
agent_config = {"configurable": {"thread_id": "agente-d-006"}}
paused = agent.invoke({"messages": [HumanMessage("Desembolsa 25.000.000 COP al cliente C-4415.")]}, agent_config)

[action] = paused["__interrupt__"][0].value["action_requests"]
print("Acción propuesta:", action["name"], action["args"])
print("Decisiones permitidas:", paused["__interrupt__"][0].value["review_configs"][0]["allowed_decisions"])

El analista aprueba. La herramienta se ejecuta y el modelo redacta la respuesta final (**1 llamada**). La decisión va en una lista, `decisions`, porque el modelo puede proponer varias herramientas en el mismo turno.

In [ ]:
approved = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), agent_config)
print(approved["messages"][-1].text)

Otra solicitud, ahora rechazada con un motivo. La herramienta **no se ejecuta** y el modelo recibe el motivo para explicárselo a quien pidió el desembolso (**unas 2 llamadas**).

In [ ]:
reject_config = {"configurable": {"thread_id": "agente-d-007"}}
agent.invoke({"messages": [HumanMessage("Desembolsa 35.000.000 COP al cliente C-3307.")]}, reject_config)
rejected = agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "El cliente tiene 75 días de mora; no se desembolsa."}]}),
    reject_config,
)

executed = [m for m in rejected["messages"] if m.type == "tool" and "Desembolsados" in m.text]
print("¿Se ejecutó la herramienta?", bool(executed))
print(rejected["messages"][-1].text)

## 6. Qué cambia con la persona en el flujo

| | Sin HITL | Con HITL |
|---|---|---|
| Quién decide una acción con efectos | El modelo, guiado por el prompt | Una persona, con el control en el código |
| Si el modelo se equivoca de monto | Se ejecuta | La persona lo ve y lo edita o rechaza antes |
| Estado mientras se espera | No aplica | Guardado por el checkpointer en su `thread_id` |
| Costo | Menor | Una pausa y, a veces, una llamada más para explicar la decisión |

Pon la pausa solo donde la acción tiene efectos difíciles de deshacer: si todo pide aprobación, las personas aprueban sin leer. Y combina capas: reglas en código para lo que es claro (el tope diario de 40 M se rechaza sin persona), aprobación humana para lo que requiere criterio y registro de cada decisión en el estado.

## Práctica breve

1. En la sección 4, edita un desembolso a un monto mayor que el tope diario. ¿Lo bloquea el grafo? ¿Dónde pondrías esa validación?
2. Añade al estado un campo `reviewer` con el nombre de quien aprobó y guárdalo en `approval`.
3. En la sección 5, añade una herramienta de solo lectura, `check_credit`, que **no** pida aprobación, y pide «Revisa la mora de C-3307 y, si está al día, desembolsa 25 M».
4. Responde a la pausa de la sección 5 con `{"type": "edit", "edited_action": {"name": "request_disbursement", "args": {...}}}` y un monto menor. ¿Qué ejecuta la herramienta?

## Referencias

- [Interrupts en LangGraph](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [Human-in-the-loop en LangChain](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- **Learning LangChain**, capítulo 8: patrones de interacción con personas.